# Experiment 4 — Predicting SPY Direction and Large Downside Moves Using Compressed Candlestick Features

**Stock Forecasting Project**

Copyright © 2026 by Tien Le


> **Public Demo Version**
>
> This notebook preserves the original data workflow, model names, training, evaluation, results, and conclusions.
> Only proprietary feature-building formulas and original feature names are omitted where applicable.


# Table of Contents

1. [Introduction](#Introduction)
2. [Motivation](#Motivation)
3. [Research Questions](#Research-Questions)
4. [Dataset Description](#Dataset-Description)
5. [Methodology](#Methodology)
6. [Data Preparation](#Data-Preparation)
7. [Feature Engineering](#Feature-Engineering)
8. [Avoiding Data Leakage](#Avoiding-Data-Leakage)
9. [Version 1: Next-Day Direction](#Version-1-Next-Day-Direction)
10. [Version 2: Large Upward Moves](#Version-2-Large-Upward-Moves)
11. [Version 3: Large Downside Moves](#Version-3-Large-Downside-Moves)
12. [Model Comparison Across Downside Targets](#Model-Comparison-Across-Downside-Targets)
13. [Results](#Results)
14. [Discussion](#Discussion)
15. [Practical Interpretation](#Practical-Interpretation)
16. [Limitations](#Limitations)
17. [Final Conclusion](#Final-Conclusion)
18. [Next Experiment](#Next-Experiment)


# Introduction

Experiments 1–3 established three important findings:

1. Classical time-series models could not meaningfully improve exact next-day SPY closing-price forecasts.
2. Machine-learning classifiers produced only modest improvements when predicting ordinary Up/Down direction.
3. Adding broader market information from VIX, QQQ, Treasury yields, and Bitcoin did not improve performance.

Experiment 4 returns the focus to SPY itself and develops a more compact, interpretable representation of candlestick structure, price action, volume, trend, momentum, and volatility.

The experiment first predicts ordinary next-day direction and then reframes the problem around unusually large market moves, with particular emphasis on large downside events.


# Motivation

Ordinary daily market direction is extremely noisy. A model may have limited value even when it improves overall accuracy by a small amount.

Large downside moves are less frequent, but they are often more important for portfolio protection and risk management. A warning model does not need to predict every daily movement correctly to be useful. It may still provide practical value if it identifies a substantial portion of the most damaging down days.

This experiment therefore asks whether simplified candlestick-style features can produce useful warning signals for large next-day declines.


# Research Questions

> Can compressed SPY candlestick, price-action, volume, and technical features improve next-day market-direction prediction?

The experiment also investigates:

1. Can the same features predict unusually large upward moves?
2. Can they detect unusually large downside moves?
3. Which target definition produces the strongest recall for downside warnings?
4. Do Logistic Regression, Decision Tree, Random Forest, or XGBoost perform best?
5. Can a simple interpretable model outperform more complicated ensemble methods?


# Dataset Description

The analysis uses daily SPY market data from Yahoo Finance beginning on January 1, 2017.

The input data include:

- Open
- High
- Low
- Close
- Adjusted Close
- Volume

The most recent year, approximately 252 trading days, is reserved as the test set. All earlier observations are used for training.

This chronological split prevents future market observations from being used to train models that predict earlier dates.


# Methodology

The experiment is divided into three stages:

1. **Ordinary direction:** predict whether tomorrow's SPY close will be above today's close.
2. **Large upward moves:** predict whether tomorrow's return will fall in the top 10% of training-period returns.
3. **Large downside moves:** predict whether tomorrow's return will fall below several downside thresholds, including the bottom 10%, 15%, 20%, 25%, 30%, and 40%.

Thresholds are calculated using the training period only. This is essential because calculating them from the complete dataset would leak information from the test period.

For rare-event targets, overall accuracy can be misleading. Therefore, special attention is given to:

- Recall
- Precision
- F1 score
- Actual target days
- Predicted warning days
- Correctly detected target days


# Data Preparation


In [1]:
# Packages
%pip install xgboost
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.tree import DecisionTreeClassifier, plot_tree, DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report)
import warnings
warnings.filterwarnings("ignore")
from IPython.display import Markdown, display

Note: you may need to restart the kernel to use updated packages.


In [2]:
# No public data-download package is required.
# The private preparation notebook creates the reproducible demo snapshot.


In [3]:
# Live download moved to PRIVATE-Prepare-Demo-Data.ipynb.


## Historical Data Window

The start date of 2017 captures several distinct market regimes, including the COVID-19 crash and recovery, the 2022 bear market, the AI-driven bull market, and recent market conditions.

Using this period provides enough history for technical features while avoiding excessive reliance on much older market behavior.


## Choosing the Historical Data Window for Prediction
When predicting tomorrow's SPY closing price, using more historical data is not always better. In many cases, 20 years of data does not provide significantly better predictions than 5–10 years of data. In fact, older data can sometimes reduce performance because market conditions, trading behavior, regulations, and economic environments change over time.

For this reason, many quantitative trading strategies are developed using approximately 3–10 years of historical data rather than the entire available history.

### Why I Chose a Start Date of 2017-01-01
I selected a start date of 2017-01-01, which provides approximately 9.5 years of market data. This period captures several important market regimes and major economic events, including:

- Trump's first presidential term (2017–2021)
- The COVID-19 market crash and recovery
- The Biden administration years
- The 2022 bear market
- The AI-driven bull market beginning in 2023
- Trump's second presidential term beginning in 2025

This time period is long enough to include multiple market environments while remaining recent enough to be relevant for forecasting current market behavior.

# Feature Engineering

This experiment replaces large collections of raw lagged candles with compressed structural features designed to summarize how a technical analyst reads a chart.

The feature groups include:

- Daily candle body, range, and shadow structure
- Gap behavior
- Short-term return and volatility patterns
- Consecutive Up/Down streaks
- Rolling price and volume context
- Trend and momentum measures
- Position relative to recent highs and lows
- Weekly and monthly candle context
- Multi-timeframe structure

The resulting dataset contains 253 predictors plus the target.

This exact feature pipeline is also used by the Final Production System, allowing Experiment 4 and the final model to share the same feature matrix in the future public-demo architecture.


## Feature Engineering Summary

In this version, we changed from using many raw lagged candle values to using compressed candlestick structure features.

Instead of giving the model every candle from the last 60 days individually, we summarized recent price behavior using features such as:

- Bullish and bearish candle ratios over recent windows
- Average candle body size
- Average upper and lower shadow size
- Average candle range
- Up-day and down-day streaks
- Higher-high, higher-low, lower-high, and lower-low streaks
- Distance from recent highs and lows
- Volume ratio and volume changes
- ATR and Bollinger Band volatility features
- Weekly and monthly candlestick context

The goal was to represent the chart more like how a trader reads it: not as isolated candles, but as trend, momentum, volatility, volume, and recent market structure.

In [4]:
# Public-demo feature loader
# The private feature formulas are intentionally not included.
from pathlib import Path

def locate_demo_file(filename):
    for path in [Path(filename), Path("data") / filename]:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"{filename} was not found. Run PRIVATE-Prepare-Demo-Data.ipynb, "
        "then place the generated CSV beside this notebook or in data/."
    )

def load_private_feature_snapshot(filename):
    snapshot = pd.read_csv(
        locate_demo_file(filename),
        parse_dates=["Date"]
    ).set_index("Date").sort_index()
    required = {"Target", "Next_Day_Return", "Market_Close"}
    missing = required.difference(snapshot.columns)
    if missing:
        raise ValueError(f"Missing snapshot columns: {sorted(missing)}")
    feature_cols = [c for c in snapshot if c.startswith("Feature_")]
    if not feature_cols:
        raise ValueError("No anonymized Feature_ columns were found.")
    return snapshot, feature_cols


In [5]:
snapshot, feature_cols = load_private_feature_snapshot("demo_features_experiment4_final.csv")
v_structure = snapshot[feature_cols + ["Target"]].dropna(subset=["Target"]).copy()
future_return_snapshot = snapshot["Next_Day_Return"].copy()
print("Private feature definitions hidden; anonymized snapshot loaded.")
print("Number of features:", len(feature_cols))


Private feature definitions hidden; anonymized snapshot loaded.
Number of features: 253


# Avoiding Data Leakage

Several safeguards are built into the experiment:

- All rolling features are based only on current and historical observations.
- The target uses the next trading day's return.
- The final row without a known future result is removed during historical evaluation.
- Train/test splitting is chronological.
- Large-move thresholds are calculated from the training period only.
- The target column is explicitly excluded from the feature matrix.

These safeguards ensure that the reported test results represent realistic out-of-sample predictions.


# Version 1: Next-Day Direction

The first model predicts whether tomorrow's SPY closing price will be above today's closing price.


In [6]:
# Experiment 4 and Final intentionally use this exact same v_structure.
print(v_structure.shape)
print(v_structure["Target"].value_counts())


(1952, 254)
Target
1.0    1074
0.0     878
Name: count, dtype: int64


In [7]:
X = v_structure.drop(columns=["Target"])
y = v_structure["Target"].astype(int)

test_start = X.index.max() - pd.DateOffset(years=1)

X_train = X[X.index < test_start]
X_test = X[X.index >= test_start]

y_train = y[y.index < test_start]
y_test = y[y.index >= test_start]

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

print("\nTrain target distribution:")
print(y_train.value_counts())

print("\nTest target distribution:")
print(y_test.value_counts())

always_up_accuracy = (y_test == 1).mean()

print(f"\nAlways Up Accuracy: {always_up_accuracy:.4f}")
print(f"Test period: {X_test.index.min().date()} to {X_test.index.max().date()}")

Train shape: (1700, 253)
Test shape : (252, 253)

Train target distribution:
Target
1    935
0    765
Name: count, dtype: int64

Test target distribution:
Target
1    139
0    113
Name: count, dtype: int64

Always Up Accuracy: 0.5516
Test period: 2025-07-09 to 2026-07-09


## Decision Tree Model

A small Decision Tree is used because it can identify nonlinear feature thresholds while remaining interpretable.

The tree is restricted with:

- `max_depth = 3`
- `min_samples_leaf = 20`
- `min_samples_split = 5`


In [8]:
tree_structure = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=20,
    min_samples_split=5,
    random_state=42
)

tree_structure.fit(X_train, y_train)

train_pred = tree_structure.predict(X_train)
test_pred = tree_structure.predict(X_test)

train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)
always_up_acc = (y_test == 1).mean()

cm = confusion_matrix(y_test, test_pred)

predicted_down = (test_pred == 0).sum()
correct_down = cm[0, 0]

print("========== Decision Tree: Compressed Candle Structure Features ==========")
print("Train Accuracy:", train_acc)
print("Test Accuracy :", test_acc)
print("Always Up Accuracy:", always_up_acc)

print("\nClassification Report:")
print(classification_report(y_test, test_pred))

print("\nConfusion Matrix:")
print(cm)

print("\nPrediction Summary:")
print("Predicted Down:", predicted_down)
print("Correctly predicted Down days:", correct_down)

print("\nObservation:")
print(
    f"The decision tree predicted {predicted_down} down days, "
    f"and {correct_down} of them were correct. "
    f"The test accuracy changed from {always_up_acc:.4f} "
    f"to {test_acc:.4f}."
)

========== Decision Tree: Compressed Candle Structure Features ==========
Train Accuracy: 0.5858823529411765
Test Accuracy : 0.5714285714285714
Always Up Accuracy: 0.5515873015873016

Classification Report:
              precision    recall  f1-score   support

           0       0.63      0.11      0.18       113
           1       0.57      0.95      0.71       139

    accuracy                           0.57       252
   macro avg       0.60      0.53      0.45       252
weighted avg       0.60      0.57      0.47       252


Confusion Matrix:
[[ 12 101]
 [  7 132]]

Prediction Summary:
Predicted Down: 19
Correctly predicted Down days: 12

Observation:
The decision tree predicted 19 down days, and 12 of them were correct. The test accuracy changed from 0.5516 to 0.5714.


In [9]:
display(Markdown(
    f"""
### Result Summary

The compressed candle-structure Decision Tree performed better than the Always Up baseline.

The test accuracy improved from **{always_up_acc:.4f}** to **{test_acc:.4f}**.

The model predicted **{predicted_down}** down days, and **{correct_down}** of them were correct.
"""
))


### Result Summary

The compressed candle-structure Decision Tree performed better than the Always Up baseline.

The test accuracy improved from **0.5516** to **0.5714**.

The model predicted **19** down days, and **12** of them were correct.


## Direction-Prediction Result

The Decision Tree achieved:

- **Train accuracy:** approximately 58.74%
- **Test accuracy:** approximately 55.95%
- **Always-Up baseline:** approximately 55.16%
- **Predicted Down days:** 28
- **Correct Down predictions:** 15

The model improved test accuracy slightly above the Always-Up baseline. More importantly, it did not simply predict Up every day. It generated 28 downside warnings, and 15 of them were followed by an actual Down day.

The result is an improvement over many models in Experiment 2, but ordinary daily direction remains difficult to predict reliably.


# Version 2: Large Upward Moves

The next target identifies returns in the top 10% of the training-period next-day return distribution.

This tests whether the compressed feature set can recognize unusually strong upward moves.


### Create new target as predicting Up next day in top 10%

In [10]:
# =====================================================
# 1.Build the features
# =====================================================
# 
# 2. Calculate next-day SPY return
# =====================================================

future_return = snapshot["Next_Day_Return"].copy()


# =====================================================
# 3. Align features and returns
# =====================================================

common_index = X.index.intersection(
    future_return.dropna().index
)

X = X.loc[common_index].copy()
future_return = future_return.loc[common_index].copy()

# Remove invalid feature values
X = X.replace([np.inf, -np.inf], np.nan)

valid_rows = (
    X.notna().all(axis=1)
    & future_return.notna()
)

X = X.loc[valid_rows].copy()
future_return = future_return.loc[valid_rows].copy()


# =====================================================
# 4. Chronological train/test split
# Use the most recent year as test data
# =====================================================

test_start = X.index.max() - pd.DateOffset(years=1)

X_train = X.loc[X.index < test_start].copy()
X_test = X.loc[X.index >= test_start].copy()

train_future_return = future_return.loc[X_train.index]
test_future_return = future_return.loc[X_test.index]


# =====================================================
# 5. Calculate Top-10%-Up threshold from TRAINING only
# =====================================================

top10_up_threshold = train_future_return.quantile(0.90)

print(
    "Top 10% Up-Day training threshold:",
    top10_up_threshold
)


# =====================================================
# 6. Apply the same training threshold to both periods
# =====================================================

y_train = (
    train_future_return >= top10_up_threshold
).astype(int)

y_test = (
    test_future_return >= top10_up_threshold
).astype(int)


# =====================================================
# 7. Optional combined dataframes
# =====================================================

train_top10 = X_train.copy()
train_top10["Target"] = y_train

test_top10 = X_test.copy()
test_top10["Target"] = y_test


# =====================================================
# 8. Inspect target distributions
# =====================================================

print("\nTrain shape:", train_top10.shape)
print("Test shape :", test_top10.shape)

print("\nTraining target counts:")
print(y_train.value_counts())

print(
    "Training positive rate:",
    y_train.mean()
)

print("\nTest target counts:")
print(y_test.value_counts())

print(
    "Test positive rate:",
    y_test.mean()
)

print(
    "\nTest period:",
    X_test.index.min().date(),
    "to",
    X_test.index.max().date()
)

Top 10% Up-Day training threshold: 0.01307958472918065

Train shape: (1700, 254)
Test shape : (252, 254)

Training target counts:
Next_Day_Return
0    1530
1     170
Name: count, dtype: int64
Training positive rate: 0.1

Test target counts:
Next_Day_Return
0    240
1     12
Name: count, dtype: int64
Test positive rate: 0.047619047619047616

Test period: 2025-07-09 to 2026-07-09


In [11]:
# =====================================================
# 8. Train Decision Tree
# =====================================================

tree_big_up = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=20,
    min_samples_split=5,
    class_weight=None,
    random_state=42
)

tree_big_up.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,3
,min_samples_split,5
,min_samples_leaf,20
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [12]:
train_pred = tree_big_up.predict(X_train)
test_pred = tree_big_up.predict(X_test)

cm = confusion_matrix(y_test, test_pred, labels=[0, 1])

train_accuracy = accuracy_score(y_train, train_pred)
test_accuracy = accuracy_score(y_test, test_pred)

precision = precision_score(y_test, test_pred, zero_division=0)
recall = recall_score(y_test, test_pred, zero_division=0)
f1 = f1_score(y_test, test_pred, zero_division=0)

always_no_accuracy = (y_test == 0).mean()

print("========== Decision Tree: Top 10% Up Days ==========")

print("Train Accuracy :", train_accuracy)
print("Test Accuracy  :", test_accuracy)
print("Always-No Accuracy:", always_no_accuracy)

print("\nPrecision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nClassification Report:")
print(classification_report(y_test, test_pred, zero_division=0))

print("\nConfusion Matrix:")
print(cm)

actual_big_up = (y_test == 1).sum()
predicted_big_up = (test_pred == 1).sum()
correct_big_up = cm[1, 1]

print("\nPrediction Summary:")
print("Actual Top-10% Up Days   :", actual_big_up)
print("Predicted Top-10% Up Days:", predicted_big_up)
print("Correct Big Up Days      :", correct_big_up)

print("\nObservation:")
print(
    f"The model predicted {predicted_big_up} Top-10% Up days, "
    f"correctly identifying {correct_big_up} of the "
    f"{actual_big_up} actual Top-10% Up days. "
    f"Accuracy changed from {always_no_accuracy:.4f} "
    f"(Always No) to {test_accuracy:.4f}."
)

========== Decision Tree: Top 10% Up Days ==========
Train Accuracy : 0.9023529411764706
Test Accuracy  : 0.9523809523809523
Always-No Accuracy: 0.9523809523809523

Precision: 0.0
Recall   : 0.0
F1 Score : 0.0

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.98       240
           1       0.00      0.00      0.00        12

    accuracy                           0.95       252
   macro avg       0.48      0.50      0.49       252
weighted avg       0.91      0.95      0.93       252


Confusion Matrix:
[[240   0]
 [ 12   0]]

Prediction Summary:
Actual Top-10% Up Days   : 12
Predicted Top-10% Up Days: 0
Correct Big Up Days      : 0

Observation:
The model predicted 0 Top-10% Up days, correctly identifying 0 of the 12 actual Top-10% Up days. Accuracy changed from 0.9524 (Always No) to 0.9524.


## Large-Up Result

The Decision Tree predicted no Top-10% Up days in the test set.

Although the reported accuracy was approximately 95.24%, that value was identical to the Always-No baseline because large upward moves were rare. Precision, recall, and F1 were all zero.

This demonstrates why accuracy alone is inappropriate for rare-event forecasting. The model achieved high accuracy by predicting that the event would never occur.


# Version 3: Large Downside Moves

The same feature set is then used to predict returns in the bottom 10% of the training-period distribution.

Large downside moves are treated as warning events.


### Let's do similarly to predict top 10% down days

In [13]:
# Decision Tree: Top 10% Down Days
# =====================================================

# -----------------------------
# 1. Create features and returns
# -----------------------------
X = v_structure.drop(
    columns=["Target"],
    errors="ignore"
).copy()

future_return = snapshot["Next_Day_Return"].copy()

# Align features and future returns
common_index = X.index.intersection(
    future_return.dropna().index
)

X = X.loc[common_index].copy()
future_return = future_return.loc[common_index].copy()

# Remove invalid rows
X = X.replace([np.inf, -np.inf], np.nan)

valid_rows = (
    X.notna().all(axis=1)
    & future_return.notna()
)

X = X.loc[valid_rows].copy()
future_return = future_return.loc[valid_rows].copy()

print("Target included as feature:", "Target" in X.columns)


# -----------------------------
# 2. Chronological split
# -----------------------------
test_start = X.index.max() - pd.DateOffset(years=1)

X_train = X.loc[X.index < test_start].copy()
X_test = X.loc[X.index >= test_start].copy()

train_future_return = future_return.loc[X_train.index]
test_future_return = future_return.loc[X_test.index]


# -----------------------------
# 3. Training-only threshold
# -----------------------------
threshold = train_future_return.quantile(0.10)

print("Top 10% Down-Day Training Threshold:", threshold)

y_train = (
    train_future_return <= threshold
).astype(int)

y_test = (
    test_future_return <= threshold
).astype(int)

print("\nTrain shape:", X_train.shape)
print("Test shape :", X_test.shape)

print("\nTraining positives:", y_train.mean())
print("Testing positives :", y_test.mean())


# -----------------------------
# 4. Train Decision Tree
# -----------------------------
tree_big_down = DecisionTreeClassifier(
    max_depth=3,
    min_samples_leaf=20,
    min_samples_split=5,
    class_weight="balanced",
    random_state=42
)

tree_big_down.fit(X_train, y_train)


# -----------------------------
# 5. Predict and evaluate
# -----------------------------
train_pred = tree_big_down.predict(X_train)
test_pred = tree_big_down.predict(X_test)

cm = confusion_matrix(
    y_test,
    test_pred,
    labels=[0, 1]
)

train_accuracy = accuracy_score(y_train, train_pred)
test_accuracy = accuracy_score(y_test, test_pred)

precision = precision_score(y_test, test_pred, zero_division=0)
recall = recall_score(y_test, test_pred, zero_division=0)
f1 = f1_score(y_test, test_pred, zero_division=0)

always_no_accuracy = (y_test == 0).mean()

actual_big_down = int((y_test == 1).sum())
predicted_big_down = int((test_pred == 1).sum())
correct_big_down = int(cm[1, 1])

print("\n========== Decision Tree: Top 10% Down Days ==========")

print("Train Accuracy :", train_accuracy)
print("Test Accuracy  :", test_accuracy)
print("Always-No Accuracy:", always_no_accuracy)

print("\nPrecision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nClassification Report:")
print(classification_report(y_test, test_pred, zero_division=0))

print("\nConfusion Matrix:")
print(cm)

print("\nPrediction Summary:")
print("Actual Top-10% Down Days   :", actual_big_down)
print("Predicted Top-10% Down Days:", predicted_big_down)
print("Correct Big Down Days      :", correct_big_down)

print("\nObservation:")
print(
    f"The model predicted {predicted_big_down} Top-10% Down days, "
    f"correctly identifying {correct_big_down} of the "
    f"{actual_big_down} actual Top-10% Down days. "
    f"Accuracy changed from {always_no_accuracy:.4f} "
    f"(Always No) to {test_accuracy:.4f}."
)

Target included as feature: False
Top 10% Down-Day Training Threshold: -0.012937642585672508

Train shape: (1700, 253)
Test shape : (252, 253)

Training positives: 0.1
Testing positives : 0.05952380952380952

========== Decision Tree: Top 10% Down Days ==========
Train Accuracy : 0.7058823529411765
Test Accuracy  : 0.7301587301587301
Always-No Accuracy: 0.9404761904761905

Precision: 0.1044776119402985
Recall   : 0.4666666666666667
F1 Score : 0.17073170731707318

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.75      0.84       237
           1       0.10      0.47      0.17        15

    accuracy                           0.73       252
   macro avg       0.53      0.61      0.50       252
weighted avg       0.91      0.73      0.80       252


Confusion Matrix:
[[177  60]
 [  8   7]]

Prediction Summary:
Actual Top-10% Down Days   : 15
Predicted Top-10% Down Days: 67
Correct Big Down Days      : 7

Observation:
The model 

In [14]:
# Create values so they automatically update when the notebook is rerun
actual_big_down = int(y_test.sum())
predicted_big_down = int((test_pred == 1).sum())
correct_big_down = int(cm[1, 1])
false_positive = int(cm[0, 1])

display(Markdown(f"""
## Conclusion

- Predicting the **Top 10% Down days** is much more challenging than predicting the daily market direction.

- The Decision Tree correctly identified **{correct_big_down} of the {actual_big_down}** actual large down days (**{recall:.1%} recall**), indicating that it learned some patterns associated with significant market declines.

- However, the model also produced **{false_positive} false alarms**, resulting in a relatively low **{precision:.1%} precision**. Most predicted large down days did not actually occur.

- Although the overall accuracy (**{test_accuracy:.1%}**) is lower than the **Always No** baseline (**{always_no_accuracy:.1%}**), accuracy is not an appropriate metric for this highly imbalanced problem. **Precision, recall, and F1-score** provide a more meaningful evaluation.

- Overall, the model shows **some ability to detect major market declines**, but it is not yet reliable enough for practical forecasting. Future work should focus on improving feature engineering and reducing false positives while maintaining the ability to detect large down days.
"""))


## Conclusion

- Predicting the **Top 10% Down days** is much more challenging than predicting the daily market direction.

- The Decision Tree correctly identified **7 of the 15** actual large down days (**46.7% recall**), indicating that it learned some patterns associated with significant market declines.

- However, the model also produced **60 false alarms**, resulting in a relatively low **10.4% precision**. Most predicted large down days did not actually occur.

- Although the overall accuracy (**73.0%**) is lower than the **Always No** baseline (**94.0%**), accuracy is not an appropriate metric for this highly imbalanced problem. **Precision, recall, and F1-score** provide a more meaningful evaluation.

- Overall, the model shows **some ability to detect major market declines**, but it is not yet reliable enough for practical forecasting. Future work should focus on improving feature engineering and reducing false positives while maintaining the ability to detect large down days.


## Top-10% Down Result

For the Top-10% Down target, the Decision Tree identified a meaningful portion of the rare downside events.

The test set contained 15 Top-10% Down days. The model detected 7 of them, producing recall of approximately **46.67%**.

The model issued more warnings than the number of actual large-down days, so precision remained relatively low. However, the ability to detect nearly half of the most severe downside events represented a more meaningful result than ordinary direction prediction or large-up prediction.


# Model Comparison Across Downside Targets

The experiment expands the analysis across several definitions of a downside event and compares multiple classifiers:

- Logistic Regression
- Decision Tree
- Random Forest
- XGBoost, when installed

The target thresholds are determined from training data only. The comparison emphasizes recall because the primary objective is to identify as many dangerous downside days as possible.


### Try different models to see which predict big Down days best

In [15]:
# Optional XGBoost
try:
    from xgboost import XGBClassifier
    has_xgb = True
except ImportError:
    has_xgb = False


# =====================================================
# 1. Prepare features and next-day returns
# =====================================================

future_return = snapshot["Next_Day_Return"].copy()

# Use v_structure features, but remove its previous Target column
base_X = v_structure.drop(
    columns=["Target"],
    errors="ignore"
).copy()

# Align features and future returns
common_index = base_X.index.intersection(future_return.dropna().index)

base_X = base_X.loc[common_index].copy()
future_return = future_return.loc[common_index].copy()

# Remove any remaining invalid values
base_X = base_X.replace([np.inf, -np.inf], np.nan)

valid_rows = base_X.notna().all(axis=1) & future_return.notna()

base_X = base_X.loc[valid_rows]
future_return = future_return.loc[valid_rows]

print("Available observations:", len(base_X))
print("Number of features:", base_X.shape[1])


# =====================================================
# 2. Fixed train/test split
# Most recent calendar year is the test period
# =====================================================

test_start = base_X.index.max() - pd.DateOffset(years=1)

X_train = base_X[base_X.index < test_start]
X_test = base_X[base_X.index >= test_start]

train_future_return = future_return.loc[X_train.index]
test_future_return = future_return.loc[X_test.index]

print("\nTrain shape:", X_train.shape)
print("Test shape :", X_test.shape)

print(
    "Test period:",
    X_test.index.min().date(),
    "to",
    X_test.index.max().date()
)


# =====================================================
# 3. Models to compare
# =====================================================

models = {
    "Decision Tree": DecisionTreeClassifier(
        max_depth=3,
        min_samples_leaf=20,
        min_samples_split=5,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        max_depth=5,
        min_samples_leaf=20,
        min_samples_split=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "Logistic Regression": LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=42
    )
}

if has_xgb:
    models["Simple XGBoost"] = XGBClassifier(
        objective="binary:logistic",
        n_estimators=100,
        learning_rate=0.03,
        max_depth=2,
        subsample=0.7,
        colsample_bytree=0.7,
        min_child_weight=10,
        gamma=1,
        reg_alpha=1,
        reg_lambda=10,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    )


# =====================================================
# 4. Target definitions
# =====================================================

target_settings = {
    "Top 5% Down": 0.05,
    "Top 10% Down": 0.10,
    "Top 15% Down": 0.15,
    "Top 20% Down": 0.20,
    "Top 25% Down": 0.25,
    "Top 30% Down": 0.30,
    "Top 35% Down": 0.35,
    "Top 40% Down": 0.40,
    "Top 45% Down": 0.45,
    "All Down Days": "all_down"
}


# =====================================================
# 5. Run all experiments
# =====================================================

results = []

for target_name, q in target_settings.items():

    # -------------------------------------------------
    # Create target
    # Threshold is calculated from TRAINING returns only
    # -------------------------------------------------
    if q == "all_down":
        threshold = 0.0

        y_train = (train_future_return < 0).astype(int)
        y_test = (test_future_return < 0).astype(int)

    else:
        threshold = train_future_return.quantile(q)

        y_train = (
            train_future_return <= threshold
        ).astype(int)

        y_test = (
            test_future_return <= threshold
        ).astype(int)

    for model_name, model_template in models.items():

        # Create a fresh copy of each model
        model = clone(model_template)

        # Train
        model.fit(X_train, y_train)

        # Predict and preserve the date index
        test_pred = pd.Series(
            model.predict(X_test),
            index=X_test.index,
            name="Prediction"
        ).astype(int)

        # -------------------------------------------------
        # Confusion matrix
        # labels=[0, 1] guarantees a 2x2 matrix
        # -------------------------------------------------
        cm = confusion_matrix(
            y_test,
            test_pred,
            labels=[0, 1]
        )

        tn, fp, fn, tp = cm.ravel()

        # -------------------------------------------------
        # Standard classification metrics
        # -------------------------------------------------
        accuracy = accuracy_score(y_test, test_pred)

        precision = precision_score(
            y_test,
            test_pred,
            zero_division=0
        )

        recall = recall_score(
            y_test,
            test_pred,
            zero_division=0
        )

        f1 = f1_score(
            y_test,
            test_pred,
            zero_division=0
        )

        always_no_accuracy = (y_test == 0).mean()

        # -------------------------------------------------
        # Target-event counts
        # -------------------------------------------------
        actual_down = int(y_test.sum())
        predicted_down = int(test_pred.sum())
        correct_down = int(tp)

        # -------------------------------------------------
        # Practical direction check
        #
        # Among every day predicted as Top-X% Down:
        # - How many actually fell at all?
        # - How many actually rose?
        # -------------------------------------------------
        predicted_down_mask = test_pred == 1

        predicted_any_down = int(
            (
                predicted_down_mask
                & (test_future_return < 0)
            ).sum()
        )

        predicted_up = int(
            (
                predicted_down_mask
                & (test_future_return >= 0)
            ).sum()
        )

        if predicted_down > 0:
            down_hit_rate = (
                predicted_any_down / predicted_down
            )
        else:
            down_hit_rate = np.nan

        # -------------------------------------------------
        # Store results using consistent column names
        # -------------------------------------------------
        results.append({
            "Target": target_name,
            "Threshold": float(threshold),
            "Model": model_name,

            "Actual Down Days": actual_down,
            "Predicted Down Days": predicted_down,
            "Correct Down Days": correct_down,

            "Predicted Days That Were Down": predicted_any_down,
            "Predicted Days That Were Up": predicted_up,
            "Down Hit Rate": down_hit_rate,

            "Precision": precision,
            "Recall": recall,
            "F1": f1,

            "Accuracy": accuracy,
            "Always-No Accuracy": always_no_accuracy,

            "True Negatives": int(tn),
            "False Positives": int(fp),
            "False Negatives": int(fn),
            "True Positives": int(tp)
        })


# =====================================================
# 6. Results table
# =====================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=["Recall", "Precision", "F1"],
    ascending=False
).reset_index(drop=True)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

display(results_df)





Available observations: 1952
Number of features: 253

Train shape: (1700, 253)
Test shape : (252, 253)
Test period: 2025-07-09 to 2026-07-09


,Target,Threshold,Model,Actual Down Days,Predicted Down Days,Correct Down Days,Predicted Days That Were Down,Predicted Days That Were Up,Down Hit Rate,Precision,Recall,F1,Accuracy,Always-No Accuracy,True Negatives,False Positives,False Negatives,True Positives
0,Top 20% Down,-0.006707,Decision Tree,33,143,26,67,76,0.468531,0.181818,0.787879,0.295455,0.507937,0.869048,102,117,7,26
1,Top 25% Down,-0.004979,Decision Tree,44,140,34,68,72,0.485714,0.242857,0.772727,0.369565,0.539683,0.825397,102,106,10,34
2,Top 40% Down,-0.001039,Decision Tree,94,153,63,72,81,0.470588,0.411765,0.670213,0.510121,0.519841,0.626984,68,90,31,63
3,Top 30% Down,-0.003223,Decision Tree,63,135,39,62,73,0.459259,0.288889,0.619048,0.393939,0.523810,0.750000,93,96,24,39
4,Top 15% Down,-0.008823,Decision Tree,25,101,15,47,54,0.465347,0.148515,0.600000,0.238095,0.619048,0.900794,141,86,10,15
5,All Down Days,0.000000,Decision Tree,113,145,67,67,78,0.462069,0.462069,0.592920,0.519380,0.507937,0.551587,61,78,46,67
6,Top 45% Down,0.000016,Decision Tree,113,143,63,63,80,0.440559,0.440559,0.557522,0.492188,0.484127,0.551587,59,80,50,63
7,Top 10% Down,-0.012938,Decision Tree,15,67,7,26,41,0.388060,0.104478,0.466667,0.170732,0.730159,0.940476,177,60,8,7
8,Top 20% Down,-0.006707,Random Forest,33,50,13,22,28,0.440000,0.260000,0.393939,0.313253,0.773810,0.869048,182,37,20,13
9,Top 25% Down,-0.004979,Random Forest,44,62,16,29,33,0.467742,0.258065,0.363636,0.301887,0.706349,0.825397,162,46,28,16


In [16]:
# Automatically find the top three Recall results
# -------------------------------------------------------

top3 = results_df.head(3).reset_index(drop=True)

r1 = top3.iloc[0]
r2 = top3.iloc[1]
r3 = top3.iloc[2]

In [17]:
display(Markdown(
    f"""
# Conclusion

Among all experiments, the models with the **highest ability to detect target down days, as measured by recall,** were:

### 🥇 Best Model
- **Target:** {r1['Target']}
- **Model:** {r1['Model']}
- **Recall:** **{r1['Recall']:.1%}**
- **Precision:** **{r1['Precision']:.1%}**
- **F1 score:** **{r1['F1']:.3f}**
- Correctly detected **{int(r1['Correct Down Days'])} of {int(r1['Actual Down Days'])}** target down days.
- Predicted **{int(r1['Predicted Down Days'])}** target down days in total.
- Of those predictions, **{int(r1['Predicted Days That Were Down'])}** were followed by an actual down day.
- **Down Hit Rate:** **{r1['Down Hit Rate']:.1%}**

### 🥈 Second-Best Model
- **Target:** {r2['Target']}
- **Model:** {r2['Model']}
- **Recall:** **{r2['Recall']:.1%}**
- **Precision:** **{r2['Precision']:.1%}**
- **F1 score:** **{r2['F1']:.3f}**
- Correctly detected **{int(r2['Correct Down Days'])} of {int(r2['Actual Down Days'])}** target down days.
- Predicted **{int(r2['Predicted Down Days'])}** target down days in total.
- Of those predictions, **{int(r2['Predicted Days That Were Down'])}** were followed by an actual down day.
- **Down Hit Rate:** **{r2['Down Hit Rate']:.1%}**

### 🥉 Third-Best Model
- **Target:** {r3['Target']}
- **Model:** {r3['Model']}
- **Recall:** **{r3['Recall']:.1%}**
- **Precision:** **{r3['Precision']:.1%}**
- **F1 score:** **{r3['F1']:.3f}**
- Correctly detected **{int(r3['Correct Down Days'])} of {int(r3['Actual Down Days'])}** target down days.
- Predicted **{int(r3['Predicted Down Days'])}** target down days in total.
- Of those predictions, **{int(r3['Predicted Days That Were Down'])}** were followed by an actual down day.
- **Down Hit Rate:** **{r3['Down Hit Rate']:.1%}**

## Overall Findings

- The best-performing model was **{r1['Model']}**, trained to predict **{r1['Target']}**.

- It achieved **{r1['Recall']:.1%} recall**, meaning that it detected **{int(r1['Correct Down Days'])} of the {int(r1['Actual Down Days'])}** actual target down days.

- It achieved **{r1['Precision']:.1%} precision**, meaning that **{r1['Precision']:.1%}** of its target-down predictions were severe enough to qualify as **{r1['Target']}**.

- The model issued **{int(r1['Predicted Down Days'])}** warnings. Of these:
  - **{int(r1['Correct Down Days'])}** correctly qualified as **{r1['Target']}**.
  - **{int(r1['Predicted Days That Were Down'])}** resulted in some type of down day, including less-severe declines.
  - **{int(r1['Predicted Days That Were Up'])}** resulted in an up or flat day.
  - The resulting **Down Hit Rate was {r1['Down Hit Rate']:.1%}**.

- These results suggest that compressed candlestick-structure features may contain useful information for identifying periods of market weakness.

- However, the model still produces false alarms and should be interpreted as an **early-warning or risk-management indicator**, not as a guaranteed trading signal.

- Such a warning may be useful when evaluating whether to reduce exposure to a daily leveraged bullish ETF such as **SPXL**, where market declines are magnified.

- A simple way to do this
"""))


# Conclusion

Among all experiments, the models with the **highest ability to detect target down days, as measured by recall,** were:

### 🥇 Best Model
- **Target:** Top 20% Down
- **Model:** Decision Tree
- **Recall:** **78.8%**
- **Precision:** **18.2%**
- **F1 score:** **0.295**
- Correctly detected **26 of 33** target down days.
- Predicted **143** target down days in total.
- Of those predictions, **67** were followed by an actual down day.
- **Down Hit Rate:** **46.9%**

### 🥈 Second-Best Model
- **Target:** Top 25% Down
- **Model:** Decision Tree
- **Recall:** **77.3%**
- **Precision:** **24.3%**
- **F1 score:** **0.370**
- Correctly detected **34 of 44** target down days.
- Predicted **140** target down days in total.
- Of those predictions, **68** were followed by an actual down day.
- **Down Hit Rate:** **48.6%**

### 🥉 Third-Best Model
- **Target:** Top 40% Down
- **Model:** Decision Tree
- **Recall:** **67.0%**
- **Precision:** **41.2%**
- **F1 score:** **0.510**
- Correctly detected **63 of 94** target down days.
- Predicted **153** target down days in total.
- Of those predictions, **72** were followed by an actual down day.
- **Down Hit Rate:** **47.1%**

## Overall Findings

- The best-performing model was **Decision Tree**, trained to predict **Top 20% Down**.

- It achieved **78.8% recall**, meaning that it detected **26 of the 33** actual target down days.

- It achieved **18.2% precision**, meaning that **18.2%** of its target-down predictions were severe enough to qualify as **Top 20% Down**.

- The model issued **143** warnings. Of these:
  - **26** correctly qualified as **Top 20% Down**.
  - **67** resulted in some type of down day, including less-severe declines.
  - **76** resulted in an up or flat day.
  - The resulting **Down Hit Rate was 46.9%**.

- These results suggest that compressed candlestick-structure features may contain useful information for identifying periods of market weakness.

- However, the model still produces false alarms and should be interpreted as an **early-warning or risk-management indicator**, not as a guaranteed trading signal.

- Such a warning may be useful when evaluating whether to reduce exposure to a daily leveraged bullish ETF such as **SPXL**, where market declines are magnified.

- A simple way to do this


# Results

The strongest downside-warning models were small Decision Trees.

Across the tested thresholds, the most meaningful results occurred for broader downside definitions such as the bottom 20% and bottom 25% of next-day returns. The strongest models detected approximately **75% to 80%** of the target downside days in the test period.

The exact ranking and values are generated automatically by the notebook from `results_df`, so the conclusion updates when the dataset is refreshed.

The key pattern is consistent:

- Small Decision Trees produced the best downside recall.
- Logistic Regression was useful for some targets but generally weaker.
- Random Forest and XGBoost did not consistently improve the warnings.
- Large upward moves were not predictable with the same success.
- Large downside targets produced the most practically meaningful signals.


# Discussion

Experiment 4 produced the strongest and most useful findings in the project so far.

The ordinary Up/Down model improved only slightly over the Always-Up baseline, confirming that next-day direction remains noisy.

The Top-10% Up model predicted no positive events, showing that high overall accuracy can conceal complete failure on a rare class.

The large-downside models were different. Small Decision Trees detected a substantial share of the target downside days, especially when the target included the bottom 20% to 25% of next-day returns.

This suggests that the compressed candlestick features may contain more information about downside risk than about ordinary direction or unusually large upside moves.

The fact that a small Decision Tree outperformed more complicated models is also important. The predictive relationships may depend on a limited number of interpretable thresholds, while larger ensembles may fit noise.


# Practical Interpretation

The downside models should be interpreted as warning systems rather than precise market-timing systems.

A warning does not guarantee that a large decline will occur. False positives remain common. However, the warnings may still be useful for controlling exposure to leveraged bullish positions such as SPXL or options with substantial downside risk.

A practical use could be:

- Reduce bullish leverage when several high-recall downside models issue warnings.
- Avoid opening new short-term leveraged bullish positions during warning periods.
- Treat the signal as one risk-management input rather than an automatic trading instruction.
- Record predictions prospectively to determine whether the historical performance continues out of sample.

The models do not establish that a profitable trading strategy exists. Transaction costs, missed upside, false warnings, and real-time execution must still be evaluated.


# Limitations

- The test period contains only approximately one year of market data.
- Large-down events are rare, so results can change substantially with a small number of observations.
- High recall is accompanied by relatively low precision and many false warnings.
- The thresholds define relative historical events rather than fixed percentage declines.
- The model does not account for transaction costs, taxes, slippage, or opportunity cost.
- Hyperparameter searches are limited.
- The experiment evaluates historical predictions rather than a fully prospective live deployment.
- Market behavior may change, causing previously useful candle patterns to weaken.


# Final Conclusion

Experiment 4 developed a compressed SPY candlestick and volume feature system and applied it to ordinary direction, large-up, and large-down prediction.

The ordinary next-day Direction Decision Tree slightly outperformed the Always-Up baseline and generated more meaningful Down predictions than many models tested previously. However, the improvement in overall accuracy remained modest.

The large-up model failed to identify any Top-10% Up days, despite its high overall accuracy. This confirmed that rare-event accuracy must be interpreted alongside recall and precision.

The most important result came from the large-downside models. Small Decision Trees produced the strongest warnings and, for the best target definitions, detected approximately 75% to 80% of the target downside days. These warnings may be useful as risk-management signals for avoiding or reducing leveraged bullish exposure during higher-risk periods.

The results also reinforce a broader lesson from the project: greater model complexity did not produce better forecasts. A small, interpretable Decision Tree performed better than Random Forest and XGBoost for the most useful downside targets.

This feature set and modeling approach are strong enough to advance toward the Final Production System. Before deployment, however, the next experiment evaluates whether a larger and more detailed feature set can improve recall or precision further.


# Next Experiment

Experiment 5 expands the candlestick and technical feature system with more detailed and complicated predictors.

The objective is to determine whether additional information improves the downside-warning models or merely introduces noise.

The experiment will compare the extended feature system with the simpler compressed features developed here, with particular emphasis on recall and precision for larger downside moves.
